# Bosch LA1-UW24 CF2 — auditorio ODEON en Pyroomacoustics

**Objetivo:** comparar la influencia de la directividad real medida (`LA1_UW24.CF2`) al inclinar la fuente **0°, −8°, −16°**. Mantendremos las 10 paredes del auditorio `Example.par`, la fuente en `(3,0,4)` m, los 25 receptores y la absorción 0,10. La ejecución se hace en **Google Colab**, no en el Ubuntu local.

**Dos etapas:** (A) validar y visualizar el campo directo a partir del CF2 sin realizar ray tracing; (B) ejecutar Pyroomacoustics 0.10.1 y exportar respuestas impulsivas.

**Precaución:** la geometría angular CF2 requiere cotejo con el sistema de ejes ODEON. Los valores relativos del CF2 por dirección no representan SPL calibrado. Este notebook no presupone que I-Simpa acepte CF2 directamente.

## Versión de ejecución (16 octubre 2026)
Esta versión activa **`EJECUTAR_SIMULACION = True`**. Por defecto ejecuta 3 orientaciones × 25 receptores con **3.000 rayos**. 
Tras descargar los resultados y verificar que estén completos, modifica `N_RAYS = 50000` en la celda de configuración y ejecuta de nuevo desde esa celda hasta el final.

**Importante:** las ganancias directas del CSV NO son SPL. Las RIR y métricas acústicas calculadas aún requieren contraste del eje angular CF2 con ODEON. No mezclar los resultados absolutos con el espectro físico Lw del Bosch sin calibración.


In [ ]:
%pip -q install "pyroomacoustics==0.10.1" numpy scipy pandas matplotlib

In [ ]:
# Se sube Bosch_CF2_Entradas.zip incluido en el paquete descargado.
from google.colab import files
from pathlib import Path
import zipfile,sys
ROOT=Path('/content/cf2_bosch')
ROOT.mkdir(parents=True,exist_ok=True)
if not (ROOT/'LA1_UW24.CF2').exists():
    uploaded=files.upload()
    assert 'Bosch_CF2_Entradas.zip' in uploaded, 'Sube Bosch_CF2_Entradas.zip'
    with zipfile.ZipFile('/content/Bosch_CF2_Entradas.zip') as z:
        z.extractall(ROOT)
sys.path.insert(0,str(ROOT))
print('Archivos:',[p.name for p in ROOT.iterdir()])

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, pyroomacoustics as pra
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt
from scipy.stats import linregress
from cf2_parser_v4 import CF2File
from cf2_directivity_pra import CF2OctaveDirectivity, receiver_attenuations
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import json,time
cf2=CF2File(ROOT/'LA1_UW24.CF2')
receivers=pd.read_csv(ROOT/'receivers_25.csv')
SRC=np.array([3.,0.,4.])
ORIENTATIONS=[0.,-8.,-16.]
OCTAVE=1000
print('PRA',pra.__version__, '| CF2:',cf2.metadata['manufacturer'],cf2.metadata['model_name'])
print('Bandas originales:',cf2.declared_frequencies)
print('Receptores:',len(receivers))
assert len(receivers)==25 and pra.__version__=='0.10.1' 

In [ ]:
# A) Chequeo del parser y sus planos polares (para 1000 Hz octava)
d=CF2OctaveDirectivity(cf2,OCTAVE)
a30=np.deg2rad(30.)
vec=np.array([[np.cos(a30),np.sin(a30),0.],[np.cos(a30),0.,np.sin(a30)]])
print('A 30°: horizontal (dB), vertical (dB):',d.relative_db(vec).round(3))
assert abs(d.relative_db([[1.,0.,0.]])[0])<1e-6
assert d.relative_db(vec)[1] < d.relative_db(vec)[0], 'Los planos se invirtieron: revisar el eje CF2'
fig,ax=plt.subplots(figsize=(8,4))
deg=np.linspace(-90,90,181)
th=np.deg2rad(deg)
vh=np.stack([np.cos(th),np.sin(th),np.zeros_like(th)],axis=1)
vv=np.stack([np.cos(th),np.zeros_like(th),np.sin(th)],axis=1)
ax.plot(deg,d.relative_db(vh),label='Horizontal')
ax.plot(deg,d.relative_db(vv),label='Vertical')
ax.set(xlabel='Ángulo fuera del eje (grados)',ylabel='Directividad relativa (dB)',title='Bosch LA1-UW24 — banda de octava 1000 Hz')
ax.grid(alpha=.3); ax.legend();plt.show()

In [ ]:
# Vista 3D del auditorio y los 25 receptores
verts=[];faces=[];names=[];wall_name=''
for raw in (ROOT/'auditorio_10_superficies.obj').read_text().splitlines():
 p=raw.split()
 if not p:continue
 if p[0]=='v':verts.append([float(x) for x in p[1:]])
 elif p[0]=='g':wall_name=p[1]
 elif p[0]=='f':faces.append([int(x)-1 for x in p[1:]]);names.append(wall_name)
verts=np.asarray(verts,float)
assert len(faces)==10
fig=plt.figure(figsize=(11,7));ax=fig.add_subplot(111,projection='3d')
ax.add_collection3d(Poly3DCollection([verts[f] for f in faces],alpha=.13,facecolor='lightblue',edgecolor='gray'))
ax.scatter(receivers.x_m,receivers.y_m,receivers.z_m,c='navy',s=20,label='25 receptores')
ax.scatter(*SRC,c='red',marker='*',s=200,label='Bosch LA1-UW24')
for el,col in [(0.,'gray'),(-8.,'orange'),(-16.,'red')]:
 t=np.deg2rad(el)
 ax.quiver(*SRC,np.cos(t),0,np.sin(t),length=6,color=col,normalize=True)
ax.set(xlabel='X (m)',ylabel='Y (m)',zlabel='Z (m)',xlim=(0,22),ylim=(-8,8),zlim=(0,10));ax.view_init(23,-66);ax.legend();plt.show()

In [ ]:
# Directividad DIRECTA hacia cada receptor (sin simular reverberación)
receivers_xyz=receivers[['x_m','y_m','z_m']].to_numpy()
att=receiver_attenuations(cf2,receivers_xyz,SRC,octave_hz=OCTAVE,orientations=ORIENTATIONS)
table=receivers.copy()
for el,a in att.items():table[f'direct_rel_{el:+g}deg_dB']=a
for el in [-8.,-16.]:table[f'Delta_{el:+g}_vs_0_dB']=att[el]-att[0.]
print(table.round(2).to_string(index=False))
table.to_csv(ROOT/'CF2_25_receptores_ganancia_directa.csv',index=False)
fig,ax=plt.subplots(figsize=(11,4))
for el in ORIENTATIONS:ax.plot(np.arange(1,26),att[el],marker='o',label=f'{el:g}°')
ax.set(xlabel='Receptor R01…R25',ylabel='Nivel direccional relativo al eje (dB)',title='Campo directo CF2 en 1000 Hz — aún NO es SPL en sala')
ax.grid(alpha=.3);ax.legend();plt.show()

### B. Ray tracing con fuente CF2 medida

**Ejecutar primero con 3.000 rayos**. El test completo con 50.000 puede tardar en Colab.

La clase `CF2OctaveDirectivity` utiliza la directividad relativa del CF2 e iguala la potencia **integrada sobre la esfera** con la OMNI de referencia (normalización). Para comparar SPL **absoluto** con ODEON será necesario incorporar el espectro real de potencia Lw del Bosch por octava y la calibración del RIR. 

**Nota clave en PRA 0.10.1:** `set_ray_tracing()` se ejecuta **antes** de `add_microphone_array()`. La directividad se asigna mediante `room.sources[0].set_directivity(...)`.


In [ ]:
# CONFIGURACION PARA EJECUTAR REALMENTE LA SIMULACION CF2
EJECUTAR_SIMULACION = True   # IMPORTANTE: False genera SOLO la tabla angular
N_RAYS = 3000               # Primera pasada; luego repetir con 50000
FS = 8000
ABSORPTION = 0.10
SCATTERING = 0.0
TIME_THRES = 8.0
RADIUS = 0.31
MAX_ORDER = 2
HIST_BIN = 0.01
SEED = 123
# Cada número de rayos tiene su propia carpeta para no mezclar los resultados
OUT = Path('/content/CF2_Resultados')/f'rays_{N_RAYS}'
OUT.mkdir(parents=True, exist_ok=True)
print('Simulación activada:', EJECUTAR_SIMULACION, '| Rayos:', N_RAYS,
      '| Orientaciones:', ORIENTATIONS, '| Destino:', OUT)
assert EJECUTAR_SIMULACION, 'No se ejecutaría ninguna simulación: cambia a True'


In [ ]:
def build_room(el):
    walls=[pra.wall_factory(verts[f].T,np.array([ABSORPTION]),np.array([SCATTERING]),name=name)
           for f,name in zip(faces,names)]
    room=pra.Room(walls,fs=FS,max_order=MAX_ORDER,ray_tracing=True,air_absorption=False)
    room.set_ray_tracing(n_rays=N_RAYS,receiver_radius=RADIUS,energy_thres=1e-7,
                         time_thres=TIME_THRES,hist_bin_size=HIST_BIN)
    room.add_source(SRC)
    room.sources[0].set_directivity(CF2OctaveDirectivity(cf2,center_hz=OCTAVE,elevation_deg=el))
    room.add_microphone_array(receivers_xyz.T)
    assert room.sources[0].directivity is not None
    assert room.rt_args['n_rays']==N_RAYS and room.rt_args['time_thres']==TIME_THRES
    assert all(abs(w.absorption[0]-ABSORPTION)<1e-9 for w in room.walls)
    return room

sos=butter(4,[OCTAVE/np.sqrt(2),OCTAVE*np.sqrt(2)],btype='bandpass',fs=FS,output='sos')
def metrics(h,receiver_xyz):
    ef=sosfiltfilt(sos,np.asarray(h,dtype=float))**2
    edc=np.cumsum(ef[::-1])[::-1]
    if not edc[0]>0:return {'T30_s':np.nan,'EDT_s':np.nan,'C80_dB':np.nan,'energia_rel_dB':np.nan}
    db=10*np.log10(np.maximum(edc/edc[0],1e-30))
    t=np.arange(len(ef))/FS
    def fit(hi,lo):
        use=(db<=hi)&(db>=lo)
        if np.count_nonzero(use)<10:return np.nan
        slope=linregress(t[use],db[use]).slope
        return -60/slope if slope<0 else np.nan
    td=np.linalg.norm(np.asarray(receiver_xyz)-SRC)/343.
    ncut=min(len(ef)-1,int((td+.080)*FS))
    return {'T30_s':fit(-5,-35),'EDT_s':fit(-.1,-10),
            'C80_dB':10*np.log10(max(ef[:ncut].sum(),1e-30)/max(ef[ncut:].sum(),1e-30)),
            'energia_rel_dB':10*np.log10(ef.sum()),'RIR_duracao_s':len(ef)/FS}
print('build_room y metrics preparados')

In [ ]:
# EJECUCIÓN REAL: tres orientaciones, 25 receptores
all_results=[]
for el in (ORIENTATIONS if EJECUTAR_SIMULACION else []):
    pra.random.seed(SEED) if hasattr(pra.random,'seed') else np.random.seed(SEED)
    room=build_room(el)
    t0=time.perf_counter();room.compute_rir();runtime=time.perf_counter()-t0
    rows=[];rirs={}
    for i,rec in receivers.iterrows():
        h=np.asarray(room.rir[i][0],dtype=np.float32)
        rirs[rec.receptor]=h
        rows.append({'receptor':rec.receptor,'elevacion_deg':el,
                     **metrics(h,rec[['x_m','y_m','z_m']].to_numpy())})
    df=pd.DataFrame(rows)
    assert len(df)==25 and set(df['receptor'])==set(receivers['receptor']), 'Faltan receptores'
    assert df['RIR_duracao_s'].notna().all(), 'Existen RIR sin duración'
    df['energia_centrada_dB']=df.energia_rel_dB-df.energia_rel_dB.mean()
    filename=f'CF2_BOSCH_1000Hz_el{el:+g}_rays{N_RAYS}'.replace('-','m').replace('+','p')
    df.to_csv(OUT/f'{filename}_25.csv',index=False)
    np.savez_compressed(OUT/f'{filename}_RIR.npz',**rirs)
    (OUT/f'{filename}_metadados.json').write_text(json.dumps({
        'speaker':'Bosch LA1-UW24','cf2_original':'LA1_UW24.CF2','frequency_octave_Hz':OCTAVE,
        'elevacion_deg':el,'number_of_rays':N_RAYS,'fs':FS,'max_order':MAX_ORDER,
        'absorption':ABSORPTION,'scattering':SCATTERING,'time_thres':TIME_THRES,
        'receiver_radius':RADIUS,'room_rt_args':str(room.rt_args),
        'runtime_seconds':runtime,'note':'Directivity power normalized; RIR units not calibrated to absolute Lw'
    },indent=2))
    all_results.append(df)
    print('elevación',el,'tempo',round(runtime,2),'T30 medio',round(df.T30_s.mean(),3),
          'C80 medio',round(df.C80_dB.mean(),3),'RIR max',round(df.RIR_duracao_s.max(),3))
assert len(all_results)==3, 'No se completaron las tres orientaciones CF2'
pd.concat(all_results).to_csv(OUT/'CF2_1000Hz_tres_orientaciones.csv',index=False)
assert len(pd.read_csv(OUT/'CF2_1000Hz_tres_orientaciones.csv'))==75
print('Éxito: 3 orientaciones × 25 receptores = 75 registros acústicos')

In [ ]:
# EXPORTACIÓN VALIDADA. No permite ZIP sólo de la tabla direccional.
import zipfile
assert EJECUTAR_SIMULACION and len(all_results)==3, 'Primero hay que ejecutar las tres simulaciones'
expected=[OUT/'CF2_1000Hz_tres_orientaciones.csv']
for el in ORIENTATIONS:
    root=f'CF2_BOSCH_1000Hz_el{el:+g}_rays{N_RAYS}'.replace('-','m').replace('+','p')
    expected += [OUT/f'{root}_25.csv', OUT/f'{root}_RIR.npz', OUT/f'{root}_metadados.json']
missing=[str(p) for p in expected if not p.is_file() or p.stat().st_size == 0]
assert not missing, f'Resultados ausentes o vacíos: {missing}'
for p in [x for x in expected if x.name.endswith('_25.csv')]:
    assert len(pd.read_csv(p))==25, f'CSV incompleto: {p}'
archive=Path(f'/content/CF2_BOSCH_SIMULADO_{N_RAYS}_raios.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    z.write(ROOT/'CF2_25_receptores_ganancia_directa.csv', arcname='CF2_25_receptores_ganancia_directa.csv')
    for f in expected: z.write(f,arcname=f.name)
with zipfile.ZipFile(archive) as z:
    assert len(z.namelist())==11, f'Cantidad inesperada de archivos: {z.namelist()}'
print('ZIP COMPLETO:',archive,'|',archive.stat().st_size,'bytes')
print('Contiene: 3 CSV por orientación, 3 RIR NPZ, 3 JSON y 1 CSV consolidado + directividad')
files.download(str(archive))
